# Vector Clocks & Causality — the numbers behind Season 2, Part 6

Companion notebook for *System Design, Plainly — Season 2, Part 6: Vector Clocks & Causality*.
Every figure in the post is computed here. Simulations are toy models on this machine, labelled as such; the queueing-style formulas are textbook results (Lamport 1978; Fidge 1988 / Mattern 1989; Dynamo, DeCandia et al. 2007).

In [ ]:
# 1. A minimal vector clock: events, merge, compare.
# Rules (Fidge 1988 / Mattern 1989): on a local event, increment your own entry;
# on receiving a message, take the element-wise max, then increment your own entry.
# Comparison: a -> b iff every entry of a <= the matching entry of b (and a != b).
# If neither dominates, the events are concurrent.

class VC:
    def __init__(self, n, i):
        self.v = [0] * n
        self.i = i
    def tick(self):
        self.v[self.i] += 1
        return list(self.v)
    def merge(self, other):
        self.v = [max(a, b) for a, b in zip(self.v, other)]
        return self.tick()

def compare(a, b):
    if a == b:
        return "equal"
    if all(x <= y for x, y in zip(a, b)):
        return "a -> b  (a happened before b)"
    if all(x >= y for x, y in zip(a, b)):
        return "b -> a  (b happened before a)"
    return "CONCURRENT  (neither saw the other)"

# Worked scenario: 3 replicas A, B, C.
p1, p2, p3 = VC(3, 0), VC(3, 1), VC(3, 2)
a1 = p1.tick()                 # A writes x=1            -> [1,0,0]
a2 = p1.tick()                 # A sends a message to B  -> [2,0,0]
b1 = p2.merge(a2)              # B receives, merges       -> [2,1,0]
b2 = p2.tick()                 # B writes y=2            -> [2,2,0]
c1 = p3.tick()                 # C writes z=3, alone     -> [0,0,1]
print("A write:", a1, "| B after merge+write:", b2, "| C independent write:", c1)
print("A vs B:", compare(a1, b2))
print("B vs C:", compare(b2, c1))
print("A vs C:", compare(a1, c1))
assert compare(a1, b2).startswith("a -> b")
assert compare(b2, c1) == "CONCURRENT  (neither saw the other)"
assert compare(a1, c1) == "CONCURRENT  (neither saw the other)"
print("OK: the clock proves B saw A, and proves C saw neither — that is the whole trick")

In [ ]:
# 2. What a Lamport scalar clock CANNOT tell you.
# Lamport (1978): one counter per process; on receive, max(counter, received) + 1.
# Guarantee is one-way only: a -> b implies L(a) < L(b), but L(a) < L(b) does NOT
# imply a -> b. Two concurrent events can get any two numbers.

class Lamport:
    def __init__(self):
        self.c = 0
    def tick(self):
        self.c += 1
        return self.c
    def receive(self, t):
        self.c = max(self.c, t) + 1
        return self.c

x, y = Lamport(), Lamport()
lx = x.tick()   # event on node X, no messages involved
ly = y.tick()   # event on node Y, no messages involved
print(f"Lamport stamps: X={lx}, Y={ly} -> looks ordered, but the events are concurrent")
# Same stamps, genuinely ordered case: Y receives X's message first.
x2, y2 = Lamport(), Lamport()
lx2 = x2.tick()
ly2 = y2.receive(lx2)
print(f"Ordered case stamps: X={lx2}, Y={ly2} -> identical-looking numbers, opposite truth")
print("A scalar cannot distinguish 'happened after' from 'happened at the same time'.")
print("That is why a vector — one counter per node — is needed to detect conflicts.")

In [ ]:
# 3. Wall-clock ordering failure: how often does skew flip the order?
# Two nodes with clocks skewed uniformly in [-100, +100] ms (typical NTP drift
# across machines; VMs are worse). Two writes to the same key, `gap` ms apart in
# REAL time. Last-write-wins on wall-clock stamps keeps the wrong one whenever
# the skew difference exceeds the gap.
import random
rng = random.Random(11)

def p_misordered(gap_ms, trials=200_000, skew=100):
    bad = 0
    for _ in range(trials):
        s_a = rng.uniform(-skew, skew)   # node A's clock offset
        s_b = rng.uniform(-skew, skew)   # node B's clock offset
        # A writes at real t, B writes at real t + gap. B is stamped earlier iff
        # (t + gap + s_b) < (t + s_a), i.e. s_a - s_b > gap.
        if s_a - s_b > gap_ms:
            bad += 1
    return bad / trials

for gap in (10, 50, 100, 200):
    print(f"writes {gap:3d} ms apart in real time -> wall clock mis-orders them "
          f"{p_misordered(gap):.1%} of the time")
# Analytic check for gap=50: skew difference is triangular on [-200, 200],
# P(D > 50) = (200-50)^2 / (2 * 200^2) = 0.28125.
assert abs(p_misordered(50) - 0.28125) < 0.01
print("OK: at 50 ms apart, ~28% of write pairs are mis-ordered by wall clock alone")
print("A 150 ms total skew (A +80, B -70) flips the order of writes 100 ms apart,")
print("so last-write-wins silently discards the newer write — with no error raised.")

In [ ]:
# 4. Conflict math: how often do two writes to one key actually collide?
# Writes arrive as a Poisson process at rate lam per second. A write conflicts
# when another write to the same key lands inside the visibility window W — the
# time before the first write has reached the other replicas. P(conflict) =
# 1 - exp(-lam * W). Expected siblings per read ~ 1 + lam * W.
import math, random
rng = random.Random(11)

def p_conflict(lam, W):
    return 1 - math.exp(-lam * W)

print("Visibility window W = 50 ms (one datacenter):")
for lam in (1, 5, 10, 20, 50, 100):
    print(f"  {lam:3d} writes/s per key -> P(conflict) = {p_conflict(lam, 0.05):.1%}   "
          f"expected siblings ~ {1 + lam * 0.05:.2f}")
print("Visibility window W = 200 ms (cross-region):")
for lam in (1, 5, 10):
    print(f"  {lam:3d} writes/s per key -> P(conflict) = {p_conflict(lam, 0.20):.1%}")
assert abs(p_conflict(10, 0.05) - 0.3935) < 1e-3
assert abs(p_conflict(100, 0.05) - 0.9933) < 1e-3
assert abs(p_conflict(10, 0.20) - 0.8647) < 1e-3

# Monte Carlo cross-check: simulate Poisson arrivals, count writes with a neighbour
# inside the window — a write at t=0 conflicts if the next write arrives within W.
def mc_conflict(lam, W, trials=20000):
    hits = 0
    for _ in range(trials):
        hits += 1 if rng.expovariate(lam) < W else 0
    return hits / trials

mc = mc_conflict(10, 0.05)
print(f"Monte Carlo check (10/s, 50 ms): {mc:.1%} vs analytic {p_conflict(10, 0.05):.1%}")
assert abs(mc - p_conflict(10, 0.05)) < 0.02
print("OK: a hot key at 10 writes/s conflicts 39% of the time; at 100/s it is 99%.")
print("That is why Dynamo returns siblings instead of picking a winner.")

In [ ]:
# 5. The size cost: a vector clock is O(N) per key version.
# Each entry: an 8-byte counter + a 16-byte node id = 24 bytes.
BYTES_PER_ENTRY = 24
for n in (3, 10, 50, 100, 1000):
    size = n * BYTES_PER_ENTRY
    lab = f"{size/1000:.1f} KB" if size >= 1000 else f"{size} B"
    print(f"N = {n:4d} nodes -> {lab} of clock per key version")
# Overhead at scale: 1M keys, N = 10 replicas.
keys = 1_000_000
print(f"1M keys x {10 * BYTES_PER_ENTRY} B (N=10): {keys * 10 * BYTES_PER_ENTRY / 1e6:.0f} MB of clocks")
# ...and a 1 KB value with a 1,000-node clock carries 24x its own size in metadata.
print(f"1 KB value, N=1000 clock: {1000 * BYTES_PER_ENTRY / 1000:.0f}x the value's size")
assert 1000 * BYTES_PER_ENTRY == 24000
# The fix, in brief: dotted version vectors give each sibling its own single
# "dot" (one counter) instead of a full vector, and entries for nodes that have
# not written recently are pruned by timestamp — bounded size, at the cost of a
# small, documented correctness risk when a pruned entry is revived. Riak made
# exactly this trade after sibling explosion in production.
print("OK: the clock is cheap at N=3 and a liability at N=1,000 — prune or shard.")

## Takeaways

1. Wall clocks cannot order events across machines: ±100 ms of NTP skew mis-orders ~28% of write pairs that are 50 ms apart in real time. Last-write-wins on a wall clock is silent data loss with a timestamp on it.
2. Lamport's scalar clock gives you a total order consistent with causality, but it cannot tell "happened after" from "happened at the same time". A vector clock can: a → b ⟺ V(a) < V(b), in both directions.
3. Conflicts are a rate problem, not a bug: P(conflict) = 1 − exp(−λ·W). At 10 writes/s per key with a 50 ms visibility window, 39% of writes are concurrent; at 100/s it is 99%. Hot keys conflict almost always — design the merge, don't hope it won't happen.
4. The clock costs O(N) per key version: 24 KB at N = 1,000, which is 24x a 1 KB value. Version vectors (one entry per replica, not per client), dotted version vectors, and pruning by timestamp keep it bounded.
5. A vector clock detects a conflict; it does not resolve one. Resolution is the application's job (merge the siblings), a CRDT's job (merge is built into the type), or an explicit, documented last-write-wins — never an accident.